In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc
from imblearn.over_sampling import SMOTE

In [ ]:
credit_card_data = pd.read_csv('/content/creditcard.csv.zip')

In [ ]:
#first 5 rows of the dataset
credit_card_data.head()

In [ ]:
credit_card_data.tail()

In [ ]:
#dataset informations
credit_card_data.info()

In [ ]:
#checking the number of missing values in each column
credit_card_data.isnull().sum()


In [ ]:
#distribution of legit transactions & fraudulent transactions
credit_card_data['Class'].value_counts()

In [ ]:
#separating the data for analysis
legit = credit_card_data[credit_card_data.Class == 0]
fraud = credit_card_data[credit_card_data.Class == 1]

In [ ]:
print(legit.shape)
print(fraud.shape)

In [ ]:
#statistical measures of the data
legit.Amount.describe()

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='Class', data=credit_card_data, palette='coolwarm')
plt.title("Class Distribution: Fraud vs Non-Fraud")
plt.xlabel("Transaction Type (0: Non-Fraud, 1: Fraud)")
plt.ylabel("Count")
plt.show()


In [ ]:
fraud.Amount.describe()

In [ ]:
plt.figure(figsize=(12,8))
sns.heatmap(credit_card_data.corr(), cmap='coolwarm', annot=False, linewidths=0.5)
plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.kdeplot(credit_card_data[credit_card_data['Class'] == 0]['Amount'], label="Non-Fraud", shade=True, color="blue")
sns.kdeplot(credit_card_data[credit_card_data['Class'] == 1]['Amount'], label="Fraud", shade=True, color="red")
plt.title("Transaction Amount Distribution")
plt.xlabel("Transaction Amount")
plt.ylabel("Density")
plt.legend()
plt.show()

In [ ]:
#compare the values for both transactions
credit_card_data.groupby('Class').mean()

In [ ]:
legit_sample = legit.sample(n=492)

In [ ]:
new_dataset = pd.concat([legit_sample, fraud], axis=0)

In [ ]:
new_dataset.head()

In [ ]:
new_dataset.tail()

In [ ]:
new_dataset['Class'].value_counts()

In [ ]:
new_dataset.groupby('Class').mean()

In [ ]:
X = new_dataset.drop(columns='Class', axis=1)
Y = new_dataset['Class']

In [ ]:
print(X)

In [ ]:
print(Y)

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=2)

In [ ]:
print(X.shape, X_train.shape, X_test.shape)

In [ ]:
model = LogisticRegression()

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_test)

plt.figure(figsize=(8,6))
plt.scatter(X_pca[:,0], X_pca[:,1], c=Y_test, cmap='coolwarm', alpha=0.5)
plt.title("PCA Visualization of Transactions")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.colorbar(label="Fraud (1) vs Normal (0)")
plt.show()

In [ ]:
credit_card_data['Time_Hours'] = credit_card_data['Time'] / 3600  # Convert seconds to hours

plt.figure(figsize=(10,5))
sns.histplot(credit_card_data[credit_card_data['Class']==1]['Time_Hours'], bins=50, color='red', kde=True)
plt.title("Fraud Transactions Over Time")
plt.xlabel("Time (Hours)")
plt.ylabel("Fraud Count")
plt.show()

In [ ]:
df_sample = credit_card_data.sample(5000)  # Reduce dataset for better performance

fig = px.scatter(df_sample, x='V1', y='V2', color='Class',
                 title="Transaction Clusters (Fraud vs Non-Fraud)",
                 labels={'Class': 'Fraud Type'},
                 opacity=0.7)
fig.show()

In [ ]:
#accuracy on training data
X_train_prediction = model.predict(X_train)
training_data_accuracy = accuracy_score(X_train_prediction, Y_train)

In [ ]:
print('Accuracy on Training data : ', training_data_accuracy)

In [ ]:
 #accuracy on test data
X_test_prediction = model.predict(X_test)
test_data_accuracy = accuracy_score(X_test_prediction, Y_test)

In [ ]:
print('Accuracy score on Test Data : ', test_data_accuracy)

In [ ]:
!pip install -q tensorflow scikit-learn pandas

In [ ]:
import pandas as pd
import numpy as np

from google.colab import files

uploaded = files.upload()
df = pd.read_csv("creditcard.csv")

print(df.head())
print(df["Class"].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop("Class", axis=1)
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Data preprocessing completed!")

In [ ]:
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=y_train
)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=256,
    class_weight={0: weights[0], 1: weights[1]},
    verbose=1
)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

probabilities = model.predict(X_test, verbose=0).ravel()
predictions = (probabilities >= 0.5).astype(int)

print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))

print("\nClassification Report:")
print(classification_report(y_test, predictions, zero_division=0))